# 108 — SmolVLA online per-chunk U20-Q75 gated s=0.3 blend

Runs one arm over the 400 standard-LIBERO indices 0–9 identities. At every decision boundary, it computes the stock chunk's `(1,2,3)/(3,1,1)` pair-weighted U20. If U20 reaches that suite's frozen indices-10–14 Q75, it executes the stock/refine 50/50 consensus projected at `s=0.3, K=3`; otherwise it executes the exact stock chunk. It replans after 10 actions. Batch size is intentionally fixed at one so a rejected projection consumes no projection RNG. Videos are off.

In [ ]:
# Install NVIDIA EGL before importing MuJoCo.
import re, subprocess
driver = subprocess.check_output(
    ['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],
    text=True).splitlines()[0].strip()
package = f"libnvidia-gl-{re.match(r'\d+', driver).group()}"
if subprocess.run('ldconfig -p | grep -q libEGL_nvidia', shell=True).returncode:
    print('Installing NVIDIA EGL package:', package)
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','--no-install-recommends',package], check=True)
subprocess.run('ldconfig -p | grep libEGL_nvidia', shell=True, check=True)

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pnp.smolvla_u20_gated_blend_experiment import (
    SMOLVLA_U20_GATED_S03_EXPERIMENT, SMOLVLA_U20_Q75_BY_SUITE,
    run_smolvla_u20_gated_s03_eval_worker)

ROLLOUT_BATCH_SIZE = 1  # required for exact conditional RNG / true online gating
print({'experiment': SMOLVLA_U20_GATED_S03_EXPERIMENT,
       'cohort': 'standard LIBERO indices 0-9', 'identities': 400,
       'arm': 'per-chunk stock U20-Q75 -> s=0.3 K3 blend, else exact stock',
       'thresholds': SMOLVLA_U20_Q75_BY_SUITE,
       'pnp_steps': [1, 2, 3], 'pnp_k_by_step': [3, 1, 1],
       'gate_horizon': 20, 'projection_s': 0.3,
       'projection_step': 7, 'projection_k': 3,
       'integration_steps': 10, 'n_action_steps': 10,
       'online_per_chunk': True, 'print_every_identities': 10,
       'video': 'off'})
run_smolvla_u20_gated_s03_eval_worker(
    rollout_batch_size=ROLLOUT_BATCH_SIZE)